# CMML3 ICA1 – Mini-project 1 


## 1. Project overview

**Aim.**  
This project models stepwise oxygen binding to haemoglobin and extends the model to investigate the effect of carbon monoxide (CO) exposure on haemoglobin function and oxygen transport.

**Main tasks completed.**

- Constructed the baseline haemoglobin–oxygen binding model in COPASI.
- Generated baseline time-course and PO2 dose-response outputs.
- Compared baseline and CO-exposed PO2 response curves.
- Investigated acute CO poisoning using an event-based simulation.
- Investigated low / normal / high 2,3-BPG-like shifts.
- Exported COPASI outputs as `.txt` files and plotted final figures in Python.


## 2. Files included in the submission package

- `COPASI_models/` – COPASI model files  
- `plot_all_figures.py` – Python script used to generate publication-style figures  
- `model_data/` – COPASI output   



## 3. Software and environment

- COPASI version: COPASI 4.46
- Python version: Python 3.10
- Python packages used: `pandas`, `matplotlib`  


## 4. Model structure in COPASI

The model was implemented in COPASI as a **single-compartment, phenomenological haemoglobin-binding model**.  
For the baseline case, haemoglobin was represented as a set of sequential oxygenation states:

- `Hb` = deoxygenated haemoglobin  
- `HbO2` = haemoglobin with 1 bound O2  
- `HbO22` = haemoglobin with 2 bound O2  
- `HbO23` = haemoglobin with 3 bound O2  
- `HbO24` = haemoglobin with 4 bound O2  

These states were connected by four reversible mass-action reactions:

- **R1:** `Hb + O2 = HbO2`
- **R2:** `HbO2 + O2 = HbO22`
- **R3:** `HbO22 + O2 = HbO23`
- **R4:** `HbO23 + O2 = HbO24`

This sequential structure was chosen to provide a simple way to represent **cooperative oxygen binding** without building a much more complex mechanistic allosteric model.

To connect the model input to a physiologically interpretable variable, dissolved oxygen was linked to oxygen partial pressure using an assignment:
`O2 = alpha_O2 × PO2`

The main readout used in the report was **Relative Saturation**, defined as the fraction of all four haem-binding sites occupied by oxygen:
`Relative_Saturation = (HbO2 + 2·HbO22 + 3·HbO23 + 4·HbO24) / (4·Total_Hb)`

where `Total_Hb` is the total haemoglobin pool in the model.

For the **CO model**, one additional reversible reaction was introduced:

- **R5:** `Hb + CO = HbCO`

and dissolved CO was linked to CO partial pressure by:
`CO = alpha_CO × PCO`

In the CO-containing models, `Total_Hb` included `HbCO`, while oxygen saturation still counted **only oxygen-occupied sites**.  
A second summary variable, `CO_Fraction = HbCO / Total_Hb`, was used to track how much haemoglobin became unavailable for oxygen transport.

The **acute CO poisoning model** used the same reaction structure as the CO model, but introduced CO dynamically with an event: the system started with `PCO = 0`, and at `t = 10` the parameter was changed to `PCO = 5` to mimic sudden exposure.

The **low-BPG** and **high-BPG** variants kept the same reaction structure as the baseline model, but modified the oxygen dissociation parameters to generate left-shifted and right-shifted oxygen-dissociation curves.


## 5. Parameter choices

### 5.1 General strategy
The reaction constants in this project were treated as phenomenological parameters, not as exact microscopic kinetic constants measured directly from one specific experiment. The aim was to choose a small set of values that reproduces the **expected qualitative physiology** of haemoglobin:
1. sigmoidal oxygen binding,
2. a normal baseline P50 close to the standard adult value (~26–27 mmHg),

### 5.2 Baseline oxygen-binding parameters
In the baseline model, the forward oxygen-binding rate was kept constant across the four oxygenation steps:
- `k1 = 0.055` for R1–R4
The reverse rates were decreased stepwise:
- **R1:** `k2 = 6.0`
- **R2:** `k2 = 2.5`
- **R3:** `k2 = 0.8`
- **R4:** `k2 = 0.25`
Keeping the forward term fixed while progressively lowering the reverse term makes later oxygen-binding steps effectively more favourable than earlier ones. This provides a simple numerical representation of **positive cooperativity**, which is consistent with the classical view that oxygen binding to one subunit increases the affinity of the remaining subunits.
The main calibration target for the baseline set was the emergent **P50**.  
Using the current exported baseline curve, the model gives an apparent **P50 ≈ 27.1 mmHg**, which is close to the commonly quoted adult haemoglobin value of about **26–27 mmHg** under standard conditions.  
Because of that, this baseline parameter set was retained as a reasonable reference condition.

### 5.3 CO reaction parameters
For the CO-containing models, the additional reaction
- `Hb + CO <-> HbCO`
was assigned:
- `k1 = 0.05`
- `k2 = 0.005`

These values were chosen to make CO binding comparatively stable in the model, so that once CO is introduced, a noticeable fraction of haemoglobin becomes trapped as `HbCO` and is no longer available for oxygen transport.
This choice is biologically motivated by the well-known fact that **CO binds haemoglobin much more strongly than oxygen** (often quoted as roughly 200–250× higher affinity).  
In addition, CO poisoning does not only reduce the number of free binding sites; it also causes the remaining oxygen-binding sites to hold O2 more tightly, producing a **left shift** of the dissociation curve. To represent that second effect in a simple way, the oxygen reverse rates in the CO model were also reduced slightly:
- **R1:** `k2 = 4.5`
- **R2:** `k2 = 1.9`
- **R3:** `k2 = 0.6`
- **R4:** `k2 = 0.18`

This does **not** claim to be a full mechanistic description of haemoglobin allostery. Rather, it is a compact way to encode the expected qualitative consequence of CO exposure: **less oxygen-carrying capacity plus increased affinity of the remaining O2-loaded species**.  

### 5.4 BPG-like parameter changes
To illustrate the expected effect of 2,3-BPG on oxygen affinity, I created two additional variants. I made the low/high BPG perturbations deliberately visible so that the shift in the dose-response curves would be easy to interpret in the report figures. For the **low-BPG-like** condition, the oxygen reverse rates were decreased relative to baseline:
- **R1:** `k2 = 4.2`
- **R2:** `k2 = 1.75`
- **R3:** `k2 = 0.56`
- **R4:** `k2 = 0.175`

For the **high-BPG-like** condition, the oxygen reverse rates were increased relative to baseline:
- **R1:** `k2 = 7.8`
- **R2:** `k2 = 3.25`
- **R3:** `k2 = 1.04`
- **R4:** `k2 = 0.325`

### 5.5 Pressure-to-ligand conversion parameters
The model used:
- `alpha_O2 = 1.3`
- `alpha_CO = 1.1`
as simple proportional conversion factors between partial pressure (`PO2`, `PCO`) and the model ligand variables (`O2`, `CO`).


## 6. Simulation settings used

### 6.1 Baseline model
For the baseline reference model (`baseline_model.cps`):
- initial `PO2 = 100`
- `alpha_O2 = 1.3`
- initial free haemoglobin pool: `Hb = 2300`
- all oxygen-bound species initially set to `0`

Two main simulation modes were used:
**(a) Baseline time-course**
- Task: `Time-Course`
- Duration: `5`
- Intervals / step number: `120`

This run was used to show that the system rapidly approaches its equilibrium state from the chosen initial condition.
**(b) Baseline PO2 scan**
- Task: `Scan`
- Scanned object: initial value of `PO2`
- Range: `0` to `120`
- Number of steps: `24`
- Distribution: linear
- Subtask: `Time-Course`
This scan generated the baseline oxygen-dissociation curve used for the P50 estimate and for comparison with the CO and BPG-shifted models.

### 6.2 CO equilibrium model
For the CO comparison model (`with_CO_model.cps`):
- initial `PO2 = 100`
- initial `PCO = 1`
- `alpha_O2 = 1.3`
- `alpha_CO = 1.1`
- initial `Hb = 2300`
- initial `HbCO = 0`

The simulation settings were kept the same as for the baseline model:
- `Time-Course`: duration `5`, intervals `120`
- `Scan`: initial `PO2` scanned from `0` to `120` with `24` linear steps
Keeping the simulation task settings identical made it easier to compare the **shape and horizontal position** of the baseline and CO curves directly.

### 6.3 Acute CO poisoning model
For the event-based poisoning model (`acute_CO_event_model.cps`), I intentionally used a more hypoxic starting condition to make the poisoning effect easier to visualise:
- initial `PO2 = 40`
- initial `PCO = 0`
- `alpha_O2 = 1.3`
- `alpha_CO = 1.1`
- initial `Hb = 2300`
- initial `HbCO = 0`

The model then applied an event:
- **Event trigger:** `Time > 10`
- **Event assignment:** `PCO = 5`

The time-course settings were:
- Task: `Time-Course`
- Duration: `40`
- Intervals / step number: `800`
- Step size: `0.05`
This setup represents a sudden CO exposure at `t = 10` and allows the post-exposure dynamics of oxygen saturation and HbCO formation to be followed over time.

### 6.4 Low- and high-BPG-like models
For the BPG perturbation models (`BPG_low_model.cps` and `BPG_high_model.cps`), all simulation-task settings were kept the same as the baseline model:
- initial `PO2 = 100`
- `alpha_O2 = 1.3`
- `Time-Course`: duration `5`, intervals `120`
- `Scan`: `PO2` from `0` to `120` with `24` linear steps

Only the oxygen dissociation parameters were changed between these models.  
This means the observed left/right shifts can be interpreted as arising from the parameter perturbation itself, rather than from different solver settings or scan ranges.


## 7. How to reproduce the figures


1. Open the COPASI model file.
2. Run the baseline time-course, baseline PO2 scan, with-CO PO2 scan, acute CO event simulation, and BPG low/high scans.
3. Export each result as a tab-delimited `.txt` file using the filenames listed above.
4. Place all `.txt` files in the same folder as `plot_all_figures.py`.
5. Run the Python script to generate the final figure `.png` files.


In [ ]:
from pathlib import Path
import subprocess
import sys

base_dir = Path.cwd()
script = base_dir / 'plot_all_figures.py'

if script.exists():
    result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print('STDERR:\n', result.stderr)
else:
    print('plot_all_figures.py not found in the current folder.')


## 8. Preview of generated figures

Run the next cell after the plotting script has completed.


In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path('figures')
figure_names = [
    'Figure_1A_baseline_timecourse.png',
    'Figure_1B_baseline_PO2_vs_saturation.png',
    'Figure_1C_noCO_vs_withCO.png',
    'Figure_2A_acute_CO_poisoning.png',
    'Figure_2B_BPG_shift.png',
    'Supplementary_Figure_S1_species_dynamics.png',
]

for name in figure_names:
    path = fig_dir / name
    if path.exists():
        display(Image(filename=str(path), width=700))
        print(name)
    else:
        print(f'Missing: {name}')


## 9. Interpretation and limitations
- The model is simplified and phenomenological rather than a full mechanistic haemoglobin allostery model.
- Some parameter values were tuned to reproduce realistic qualitative behaviour rather than being fitted to a full experimental dataset.
- The acute CO poisoning simulation was designed to make the dynamic effect visible in the model output.
- The event-based simulation is useful for illustrating the direction of the effect, even if the challenge magnitude is simplified.
